### 10MIN Buckets (Real-Time Dashboard Inputs)
1. Load the raw 10-minute CSVs.


2. Standardize time zones to Manila Time (PHT).


3. Keep the required fields: PM1, PM2.5, PM10, CO2, TVOC, Temperature, Humidity.


4. Filter out impossible readings per field (bounds check).


5. Drop stuck sensor values (flatline) for key fields.


6. Remove localized noise spikes (rolling z-score) for PM columns.


7. Aggregate to 1-hour blocks for modeling consistency (optional).

### 1HR Buckets (72-Hour Forecast Inputs)
1. Load the historical 1-hour CSVs.


2. Standardize time zones to Manila Time (PHT).


3. Keep the required fields: PM1, PM2.5, PM10, Temperature, Humidity.


4. Filter out impossible readings and stuck values.



In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Resolve data folder from workspace root or notebook folder
DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("..") / "data"

RAW_DIR = DATA_DIR / "raw"
if not RAW_DIR.exists():
    RAW_DIR = DATA_DIR

OUTPUT_DIR = DATA_DIR / "cleaned"
MIN_DIR = OUTPUT_DIR / "min-buckets"
HOUR_DIR = OUTPUT_DIR / "hour-buckets"

MIN_DIR.mkdir(parents=True, exist_ok=True)
HOUR_DIR.mkdir(parents=True, exist_ok=True)


def find_pm25_col(columns: list[str]) -> str:
    pm_cols = [c for c in columns if str(c).startswith("PM2.5")]
    if not pm_cols:
        raise ValueError("No PM2.5 column found.")

    preferred = [
        c for c in pm_cols
        if "correct" in c.lower() or c.lower().endswith(" c") or "corr" in c.lower()
    ]
    return preferred[0] if preferred else pm_cols[0]


def find_timestamp_col(columns: list[str]) -> str:
    for candidate in ["UTC Date/Time", "timestamp", "Local Date/Time"]:
        if candidate in columns:
            return candidate
    raise ValueError("No timestamp column found.")


def clean_10min_df(df: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    pm_col = find_pm25_col(df.columns.tolist())
    ts_col = find_timestamp_col(df.columns.tolist())

    df = df.copy()
    df["timestamp"] = pd.to_datetime(df[ts_col], utc=True, errors="coerce")
    df["timestamp"] = df["timestamp"].dt.tz_convert("Asia/Manila")
    df = df.dropna(subset=["timestamp", pm_col]).sort_values("timestamp")

    # Step 2: remove implausible values
    df = df[(df[pm_col] >= 0) & (df[pm_col] <= 500)]

    # Step 3: drop runs of unchanged values (>= 4 consecutive readings)
    pm_round = df[pm_col].round(2)
    run_id = (pm_round != pm_round.shift()).cumsum()
    run_len = pm_round.groupby(run_id).transform("size")
    df = df[run_len < 4]

    # Step 4: rolling outlier removal (2-hour window = 12x10min)
    window = 12
    rolling_median = df[pm_col].rolling(window=window, min_periods=6, center=True).median()
    rolling_std = df[pm_col].rolling(window=window, min_periods=6, center=True).std()
    z = (df[pm_col] - rolling_median).abs() / rolling_std
    df.loc[z > 3, pm_col] = np.nan
    df = df.dropna(subset=[pm_col])

    return df, pm_col


def aggregate_hourly(df: pd.DataFrame, pm_col: str) -> pd.DataFrame:
    df = df.copy()
    df["hour"] = df["timestamp"].dt.floor("H")

    counts = df.groupby("hour")[pm_col].count()
    keep_hours = counts[counts >= 5].index
    df = df[df["hour"].isin(keep_hours)]

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    agg_map = {col: "mean" for col in numeric_cols}

    non_numeric = [
        c for c in df.columns
        if c not in numeric_cols and c not in ["timestamp", "hour"]
    ]
    agg_map.update({col: "first" for col in non_numeric})

    hourly = df.groupby("hour", as_index=False).agg(agg_map)
    hourly = hourly.rename(columns={"hour": "timestamp"})
    return hourly


summary = []
for file_path in sorted(RAW_DIR.glob("*-10MIN.csv")):
    df_raw = pd.read_csv(file_path, encoding="utf-8", encoding_errors="replace")
    df_clean, pm_col = clean_10min_df(df_raw)
    df_hourly = aggregate_hourly(df_clean, pm_col)

    clean_name = file_path.stem + "-clean.csv"
    hourly_name = file_path.stem.replace("-10MIN", "-10MIN-Agg") + ".csv"

    df_clean.to_csv(MIN_DIR / clean_name, index=False)
    df_hourly.to_csv(HOUR_DIR / hourly_name, index=False)

    summary.append(
        (file_path.name, len(df_raw), len(df_clean), len(df_hourly))
    )

summary_df = pd.DataFrame(
    summary,
    columns=["file", "raw_rows", "clean_10min_rows", "hourly_rows"],
)
summary_df

C:\Users\Von\AppData\Local\Temp\ipykernel_5576\3186764657.py:72: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["hour"] = df["timestamp"].dt.floor("H")
C:\Users\Von\AppData\Local\Temp\ipykernel_5576\3186764657.py:72: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["hour"] = df["timestamp"].dt.floor("H")
C:\Users\Von\AppData\Local\Temp\ipykernel_5576\3186764657.py:72: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["hour"] = df["timestamp"].dt.floor("H")
C:\Users\Von\AppData\Local\Temp\ipykernel_5576\3186764657.py:72: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["hour"] = df["timestamp"].dt.floor("H")
C:\Users\Von\AppData\Local\Temp\ipykernel_5576\3186764657.py:72: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df[

,file,raw_rows,clean_10min_rows,hourly_rows
0,cubao-10MIN.csv,19898,19119,3142
1,lawton-10MIN.csv,18882,18533,3078
2,makati-10MIN.csv,2223,2220,367
3,pasay-10MIN.csv,4936,4733,765
4,SDG-10MIN.csv,663,654,101


STEP 2 - CLEAN HISTORICAL 1HR DATA

In [2]:
def clean_1hr_df(df: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    pm_col = find_pm25_col(df.columns.tolist())
    ts_col = find_timestamp_col(df.columns.tolist())

    df = df.copy()
    df["timestamp"] = pd.to_datetime(df[ts_col], utc=True, errors="coerce")
    df["timestamp"] = df["timestamp"].dt.tz_convert("Asia/Manila")
    df = df.dropna(subset=["timestamp", pm_col]).sort_values("timestamp")

    # Remove implausible values
    df = df[(df[pm_col] >= 0) & (df[pm_col] <= 500)]

    # Drop runs of unchanged values (>= 3 consecutive hourly readings)
    pm_round = df[pm_col].round(2)
    run_id = (pm_round != pm_round.shift()).cumsum()
    run_len = pm_round.groupby(run_id).transform("size")
    df = df[run_len < 3]

    return df, pm_col


summary_1hr = []
for file_path in sorted(RAW_DIR.glob("*-1HR.csv")):
    df_raw = pd.read_csv(file_path, encoding="utf-8", encoding_errors="replace")
    df_clean, pm_col = clean_1hr_df(df_raw)

    clean_name = file_path.stem + "-clean.csv"
    df_clean.to_csv(HOUR_DIR / clean_name, index=False)

    summary_1hr.append(
        (file_path.name, len(df_raw), len(df_clean))
    )

summary_1hr_df = pd.DataFrame(
    summary_1hr,
    columns=["file", "raw_rows", "clean_rows"],
)
summary_1hr_df

,file,raw_rows,clean_rows
0,cubao-1HR.csv,3332,3253
1,lawton-1HR.csv,3162,3141
2,makati-1HR.csv,376,376
3,pasay-1HR.csv,847,826
4,SDG-1HR.csv,127,126
